In [1]:
print("NS")

NS


Why--> Parralel Processsing

Running no.of fuctions at a time

In [8]:
import time
import random
from concurrent.futures import ThreadPoolExecutor

def my_func(x):
    
    # waiting = time.sleep(random.randint(1, 3))

    wait = random.randint(1, 3)
    time.sleep(wait)

    print(f"Time taking is : {wait} secondss")
    
my_func(101)    

Time taking is : 3 secondss


In [9]:
import time
import random
from concurrent.futures import ThreadPoolExecutor

tables = ["Orders", "Products", "Customers","Reviews", "Cancelled_Orders"]

def my_func(x):
    
    # waiting = time.sleep(random.randint(1, 3))

    wait = random.randint(1, 3)
    time.sleep(wait)

    print(f"i am item from table:- {x} ,taking time is:- {wait} secondss")
    
for x in tables:
    my_func(x)

i am item from table:- Orders ,taking time is:- 3 secondss
i am item from table:- Products ,taking time is:- 3 secondss
i am item from table:- Customers ,taking time is:- 2 secondss
i am item from table:- Reviews ,taking time is:- 3 secondss
i am item from table:- Cancelled_Orders ,taking time is:- 1 secondss


# # To Sovle above more waiting time by multi-threading

In [12]:
import time
import random
from concurrent.futures import ThreadPoolExecutor

tables = ["Orders", "Products", "Customers","Reviews", "Cancelled_Orders"]

def my_func(x):
    
    wait = random.randint(1, 3)
    time.sleep(wait)
    print(f"i am item from table:- {x} ,taking time is:- {wait} secondss..")
    
# here how solve a problem by using ---> multi threading    
    
with ThreadPoolExecutor(max_workers=len(tables)) as executor:
    executor.map(my_func, tables)

i am item from table:- Cancelled_Orders ,taking time is:- 1 secondss..
i am item from table:- Orders ,taking time is:- 2 secondss..
i am item from table:- Reviews ,taking time is:- 2 secondss..
i am item from table:- Customers ,taking time is:- 2 secondss..
i am item from table:- Products ,taking time is:- 3 secondss..


i am item from table:- Cancelled_Orders ,taking time is:- 1 secondss..i am item from table:- Products ,taking time is:- 1 secondss..
i am item from table:- Orders ,taking time is:- 1 secondss..

i am item from table:- Reviews ,taking time is:- 2 secondss..
i am item from table:- Customers ,taking time is:- 3 secondss..


---

# 🔹 What is Multithreading?

Multithreading allows a program to **run multiple threads (lightweight tasks) concurrently within the same process**.

👉 Think:

* One program
* Multiple tasks running *at the same time* (or appearing to)

---

# 🔹 Why Multithreading in Data Engineering?

Used when tasks are:

* Waiting on I/O (APIs, DB, files, Kafka, S3)
* Network-heavy operations

👉 Example:

* Fetch data from 10 APIs simultaneously instead of sequentially

---

# 🔴 Important Concept: GIL (Very Important in Interviews)

Python has a **Global Interpreter Lock (GIL)**

👉 Meaning:

* Only **one thread executes Python bytecode at a time**
* So:

  * ❌ Not good for CPU-heavy tasks
  * ✅ Good for I/O-bound tasks

---

# 🔹 Basic Example (Without Threading)

```python
import time

def task(name):
    print(f"Start {name}")
    time.sleep(2)  # simulate I/O wait
    print(f"End {name}")

task("Task 1")
task("Task 2")
```

👉 Output time ≈ **4 seconds** (runs sequentially)

---

# 🔹 Multithreading Example

```python
import threading
import time

def task(name):
    print(f"Start {name}")
    time.sleep(2)
    print(f"End {name}")

# Create threads
t1 = threading.Thread(target=task, args=("Task 1",))
t2 = threading.Thread(target=task, args=("Task 2",))

# Start threads
t1.start()
t2.start()

# Wait for completion
t1.join()
t2.join()
```

👉 Output time ≈ **2 seconds** (runs concurrently)

---

# 🔹 Real Data Engineering Example

### Fetch data from multiple APIs

```python
import threading
import requests

urls = [
    "https://api.github.com",
    "https://jsonplaceholder.typicode.com/posts",
    "https://jsonplaceholder.typicode.com/users"
]

def fetch(url):
    response = requests.get(url)
    print(f"{url} -> {response.status_code}")

threads = []

for url in urls:
    t = threading.Thread(target=fetch, args=(url,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()
```

👉 Faster API ingestion pipeline 🚀

---

# 🔹 Thread Lifecycle

1. New → thread created
2. Runnable → ready to run
3. Running → executing
4. Blocked → waiting (I/O, lock)
5. Terminated

---

# 🔹 Synchronization (Very Important)

When multiple threads access shared data → issues occur

## ❌ Problem (Race Condition)

```python
import threading

counter = 0

def increment():
    global counter
    for _ in range(100000):
        counter += 1

t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)

t1.start()
t2.start()

t1.join()
t2.join()

print(counter)  # ❌ unpredictable result
```

---

## ✅ Solution: Lock

```python
import threading

counter = 0
lock = threading.Lock()

def increment():
    global counter
    for _ in range(100000):
        with lock:  # critical section
            counter += 1

t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)

t1.start()
t2.start()

t1.join()
t2.join()

print(counter)  # ✅ correct result
```

---

# 🔹 ThreadPoolExecutor (Best Practice)

Cleaner and widely used in production

```python
from concurrent.futures import ThreadPoolExecutor
import time

def task(n):
    time.sleep(2)
    return f"Task {n} done"

with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(task, [1, 2, 3, 4])

for r in results:
    print(r)
```

---

# 🔹 Multithreading vs Multiprocessing

| Feature  | Multithreading | Multiprocessing |
| -------- | -------------- | --------------- |
| GIL      | Affected       | Not affected    |
| Best for | I/O tasks      | CPU tasks       |
| Memory   | Shared         | Separate        |
| Speed    | Faster for I/O | Faster for CPU  |

---

# 🔹 When to Use in Data Engineering

Use multithreading for:

* API calls (parallel ingestion)
* Reading multiple files
* Web scraping
* Kafka consumers
* Database queries (parallel reads)

Avoid for:

* Heavy ML computations → use multiprocessing

---

# 🔥 Common Interview Questions

**1. What is GIL?**
👉 Lock that allows only one thread to execute Python bytecode at a time

**2. Why still use multithreading?**
👉 Because I/O operations release GIL

**3. Difference between thread & process?**
👉 Threads share memory, processes don’t

**4. What is race condition?**
👉 Multiple threads modifying shared data incorrectly

**5. How to avoid race condition?**
👉 Use locks, semaphores

---

# 🔹 5-Line Interview Answer

> Multithreading in Python allows concurrent execution of multiple threads within a process. Due to the GIL, it is best suited for I/O-bound tasks rather than CPU-bound tasks. It is widely used in data engineering for parallel API calls, file processing, and data ingestion. Synchronization mechanisms like locks are used to avoid race conditions. For better scalability, ThreadPoolExecutor is commonly used.

---